In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import mlflow

In [0]:
# Pfad zum hochgeladenen File
DATA_PATH = "/Volumes/workspace/default/data_kpi/kpi_qa_features_2025_05.csv"

In [0]:
df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)
df = df.withColumn("ts_utc", F.to_timestamp("ts_utc"))

In [0]:
print(f"Zeilen geladen: {df.count():,}")
print(f"Einzigartige Detektoren: {df.select('det_id15').distinct().count():,}")
df.printSchema()


Zeilen geladen: 581,594
Einzigartige Detektoren: 548
root
 |-- det_id15: long (nullable = true)
 |-- ts_utc: timestamp (nullable = true)
 |-- row_count: integer (nullable = true)
 |-- missing_rate: double (nullable = true)
 |-- duplicate_rate: double (nullable = true)
 |-- freshness_lag_h: integer (nullable = true)



In [0]:
# Lokale Stunde für Gruppenbildung (pro Detektor × Stunde des Tages)
df = df.withColumn("hour_local",
    F.hour(F.from_utc_timestamp("ts_utc", "Europe/Berlin")))

feature_cols = ["row_count", "missing_rate", "duplicate_rate", "freshness_lag_h"]

# Schritt 1: Median pro (Detektor, Stunde)
stats = df.groupBy("det_id15", "hour_local").agg(
    *[F.percentile_approx(c, 0.5).alias(f"{c}_med") for c in feature_cols]
)
df2 = df.join(stats, ["det_id15", "hour_local"], "left")

# Schritt 2: Absolute Abweichungen vom Median
for c in feature_cols:
    df2 = df2.withColumn(f"{c}_abs_dev",
        F.abs(F.col(c) - F.col(f"{c}_med")))

# Schritt 3: MAD (Median der absoluten Abweichungen) pro Gruppe
mad_stats = df2.groupBy("det_id15", "hour_local").agg(
    *[F.percentile_approx(f"{c}_abs_dev", 0.5).alias(f"{c}_mad")
      for c in feature_cols]
)
df3 = df2.join(mad_stats, ["det_id15", "hour_local"], "left")

# Schritt 4: Robuste Z-Scores (MAD × 1.4826 = normalisierte Standardabweichung)
EPS = 1e-6
for c in feature_cols:
    mad_scaled = F.col(f"{c}_mad") * 1.4826
    df3 = df3.withColumn(f"z_{c}",
        F.when(mad_scaled > EPS,
               F.abs(F.col(c) - F.col(f"{c}_med")) / mad_scaled)
        .otherwise(F.lit(0.0)))

# Schritt 5: Anomalie-Score = Maximum aller Z-Scores
z_cols = [f"z_{c}" for c in feature_cols]
df3 = df3.withColumn("anomaly_score", F.greatest(*[F.col(c) for c in z_cols]))
df3 = df3.withColumn("is_anomaly", F.col("anomaly_score") >= 4.0)

df3.select("det_id15", "ts_utc", "anomaly_score", "is_anomaly").show(10)


+---------------+-------------------+-------------+----------+
|       det_id15|             ts_utc|anomaly_score|is_anomaly|
+---------------+-------------------+-------------+----------+
|100101010000167|2025-04-17 05:00:00|          0.0|     false|
|100101010022496|2025-05-13 05:00:00|          0.0|     false|
|100101010033614|2025-05-06 00:00:00|          0.0|     false|
|100101010033614|2025-05-05 18:00:00|          0.0|     false|
|100101010063421|2025-05-24 03:00:00|          0.0|     false|
|100101010063421|2025-05-24 01:00:00|          0.0|     false|
|100101010063421|2025-05-23 23:00:00|          0.0|     false|
|100101010063421|2025-05-24 02:00:00|          0.0|     false|
|100101010075444|2025-05-05 03:00:00|          0.0|     false|
|100101010075444|2025-05-05 06:00:00|          0.0|     false|
+---------------+-------------------+-------------+----------+
only showing top 10 rows


In [0]:
# Nur Monats-Daten (ohne Lookback)
month_df = df3.filter(
    (F.col("ts_utc") >= "2025-04-30 22:00:00") &
    (F.col("ts_utc") <  "2025-05-31 22:00:00")
)

total     = month_df.count()
anomalies = month_df.filter(F.col("is_anomaly") == True).count()
rate      = round(anomalies / total, 4) if total > 0 else 0.0

print(f"Zeilen gesamt:  {total:,}")
print(f"Anomalien:      {anomalies:,}  ({rate*100:.1f}%)")
print(f"Erwarteter Wert: ~44.379 Anomalien (11,1%) — wie im lokalen MLflow")

# MLflow Tracking (in Databricks nativ verfügbar)
with mlflow.start_run(run_name="zscore_pyspark_2025_05"):
    mlflow.log_params({
        "month_key":   "2025_05",
        "model":       "robust_zscore_mad",
        "framework":   "PySpark",
        "platform":    "Databricks Community Edition",
        "z_threshold": 4.0,
        "rows_total":  total,
    })
    mlflow.log_metrics({
        "rows_scored":   total,
        "anomaly_count": anomalies,
        "anomaly_rate":  rate,
    })
    print("\nRun in Databricks MLflow geloggt.")
    print("Sichtbar unter: AI/ML → Experiments")


Zeilen gesamt:  400,603
Anomalien:      0  (0.0%)
Erwarteter Wert: ~44.379 Anomalien (11,1%) — wie im lokalen MLflow

Run in Databricks MLflow geloggt.
Sichtbar unter: AI/ML → Experiments


In [0]:
# Korrekte Variante: Baseline getrennt vom Scoring-Zeitraum
baseline = df.filter(F.col("ts_utc") < "2025-04-30 22:00:00")
score_df  = df.filter(
    (F.col("ts_utc") >= "2025-04-30 22:00:00") &
    (F.col("ts_utc") <  "2025-05-31 22:00:00")
)

# Statistiken NUR aus dem Lookback berechnen
stats2 = baseline.groupBy("det_id15", "hour_local").agg(
    *[F.percentile_approx(c, 0.5).alias(f"{c}_med") for c in feature_cols]
)
df_scored = score_df.join(stats2, ["det_id15", "hour_local"], "left")

for c in feature_cols:
    df_scored = df_scored.withColumn(f"{c}_abs_dev",
        F.abs(F.col(c) - F.col(f"{c}_med")))

mad2 = baseline.join(stats2, ["det_id15","hour_local"],"left")
for c in feature_cols:
    mad2 = mad2.withColumn(f"{c}_abs_dev",
        F.abs(F.col(c) - F.col(f"{c}_med")))

mad_stats2 = mad2.groupBy("det_id15","hour_local").agg(
    *[F.percentile_approx(f"{c}_abs_dev", 0.5).alias(f"{c}_mad") for c in feature_cols]
)
df_scored = df_scored.join(mad_stats2, ["det_id15","hour_local"], "left")

EPS = 1e-6
for c in feature_cols:
    mad_sc = F.col(f"{c}_mad") * 1.4826
    df_scored = df_scored.withColumn(f"z_{c}",
        F.when(mad_sc > EPS, F.abs(F.col(c) - F.col(f"{c}_med")) / mad_sc)
        .otherwise(F.lit(0.0)))

df_scored = df_scored.withColumn("anomaly_score",
    F.greatest(*[F.col(f"z_{c}") for c in feature_cols]))
df_scored = df_scored.withColumn("is_anomaly", F.col("anomaly_score") >= 4.0)

total2     = df_scored.count()
anomalies2 = df_scored.filter("is_anomaly = true").count()
rate2      = round(anomalies2/total2, 4) if total2 > 0 else 0.0
print(f"Korrekte Variante: {anomalies2:,} Anomalien von {total2:,} ({rate2*100:.1f}%)")


Korrekte Variante: 0 Anomalien von 400,603 (0.0%)


In [0]:
# Globale Fallback-Statistiken aus dem Baseline-Zeitraum
global_stats = baseline.agg(
    *[F.percentile_approx(c, 0.5).alias(f"{c}_med_glob") for c in feature_cols]
)

# Absolute Abweichungen für globale MAD
baseline_with_glob = baseline
for c in feature_cols:
    glob_med = global_stats.collect()[0][f"{c}_med_glob"]
    baseline_with_glob = baseline_with_glob.withColumn(
        f"{c}_abs_dev_glob", F.abs(F.col(c) - F.lit(glob_med)))

global_mad = baseline_with_glob.agg(
    *[F.percentile_approx(f"{c}_abs_dev_glob", 0.5).alias(f"{c}_mad_glob")
      for c in feature_cols]
)

# Werte extrahieren
glob_vals = {**global_stats.collect()[0].asDict(),
             **global_mad.collect()[0].asDict()}

# Z-Score mit Fallback: Gruppe → global
for c in feature_cols:
    med = F.coalesce(F.col(f"{c}_med"), F.lit(glob_vals[f"{c}_med_glob"]))
    mad = F.coalesce(F.col(f"{c}_mad"), F.lit(glob_vals[f"{c}_mad_glob"]))
    mad_sc = mad * 1.4826
    df_scored = df_scored.withColumn(f"z_{c}",
        F.when(mad_sc > EPS, F.abs(F.col(c) - med) / mad_sc)
        .otherwise(F.lit(0.0)))

df_scored = df_scored.withColumn("anomaly_score",
    F.greatest(*[F.col(f"z_{c}") for c in feature_cols]))
df_scored = df_scored.withColumn("is_anomaly",
    F.col("anomaly_score") >= 4.0)

total3     = df_scored.count()
anomalies3 = df_scored.filter(F.col("is_anomaly")).count()
rate3      = round(anomalies3 / total3, 4) if total3 > 0 else 0.0
print(f"Mit globalem Fallback: {anomalies3:,} Anomalien von {total3:,} ({rate3*100:.1f}%)")
print(f"Erwarteter Wert: ~44.379 (11,1%)")


Mit globalem Fallback: 0 Anomalien von 400,603 (0.0%)
Erwarteter Wert: ~44.379 (11,1%)


In [0]:
# Diagnose: Wie verteilen sich die Z-Scores wirklich?
print("Z-Score Statistiken:")
df_scored.select("anomaly_score").describe().show()

quantiles = df_scored.select("anomaly_score").approxQuantile(
    "anomaly_score", [0.90, 0.95, 0.99, 0.999, 1.0], 0.001)
print(f"90. Perzentil: {quantiles[0]:.4f}")
print(f"95. Perzentil: {quantiles[1]:.4f}")
print(f"99. Perzentil: {quantiles[2]:.4f}")
print(f"99.9. Perzentil: {quantiles[3]:.4f}")
print(f"Maximum: {quantiles[4]:.4f}")

# Zum Vergleich: wie viele Zeilen hätten bei threshold=1.0 eine Anomalie?
for thr in [1.0, 2.0, 3.0, 4.0]:
    count = df_scored.filter(F.col("anomaly_score") >= thr).count()
    print(f"threshold={thr}: {count:,} Anomalien ({count/total3*100:.1f}%)")


Z-Score Statistiken:
+-------+--------------------+
|summary|       anomaly_score|
+-------+--------------------+
|  count|              400603|
|   mean|0.020972048194632404|
| stddev| 0.15255259812442412|
|    min|                 0.0|
|    max|  1.3489835424274925|
+-------+--------------------+

90. Perzentil: 0.0000
95. Perzentil: 0.0000
99. Perzentil: 1.3490
99.9. Perzentil: 1.3490
Maximum: 1.3490
threshold=1.0: 4,212 Anomalien (1.1%)
threshold=2.0: 0 Anomalien (0.0%)
threshold=3.0: 0 Anomalien (0.0%)
threshold=4.0: 0 Anomalien (0.0%)


In [0]:
# Fix: MAD=0-Logik aus dem Original-Script nachbilden
for c in feature_cols:
    med = F.coalesce(F.col(f"{c}_med"), F.lit(glob_vals[f"{c}_med_glob"]))
    mad_raw = F.coalesce(F.col(f"{c}_mad"), F.lit(glob_vals[f"{c}_mad_glob"]))
    mad_sc = mad_raw * 1.4826

    df_scored = df_scored.withColumn(f"z_{c}",
        F.when(mad_sc > EPS,
               F.abs(F.col(c) - med) / mad_sc)           # normaler Z-Score
        .when(F.abs(F.col(c) - med) <= EPS,
               F.lit(0.0))                                 # MAD=0, Wert=Median → normal
        .otherwise(F.lit(999.0))                           # MAD=0, Wert≠Median → Anomalie!
    )

df_scored = df_scored.withColumn("anomaly_score",
    F.greatest(*[F.col(f"z_{c}") for c in feature_cols]))
df_scored = df_scored.withColumn("is_anomaly",
    F.col("anomaly_score") >= 4.0)

total4     = df_scored.count()
anomalies4 = df_scored.filter(F.col("is_anomaly")).count()
rate4      = round(anomalies4 / total4, 4) if total4 > 0 else 0.0
print(f"Korrigiert: {anomalies4:,} Anomalien von {total4:,} ({rate4*100:.1f}%)")
print(f"Erwarteter Wert: ~44.379 (11,1%)")


Korrigiert: 54,112 Anomalien von 400,603 (13.5%)
Erwarteter Wert: ~44.379 (11,1%)


In [0]:
# MLflow Tracking — finaler Run mit korrektem MAD=0-Fix
with mlflow.start_run(run_name="zscore_pyspark_fixed_2025_05"):
    mlflow.log_params({
        "month_key":        "2025_05",
        "model":            "robust_zscore_mad",
        "framework":        "PySpark",
        "platform":         "Databricks Community Edition",
        "z_threshold":      4.0,
        "mad_zero_fix":     True,   # MAD=0 → 999 statt 0
        "fallback_levels":  2,      # Gruppe + Global
    })
    mlflow.log_metrics({
        "rows_scored":     total4,
        "anomaly_count":   anomalies4,
        "anomaly_rate":    rate4,
        "local_reference": 0.111,   # Referenzwert lokale Version
        "deviation_pct":   round(abs(rate4 - 0.111) / 0.111 * 100, 1),
    })
    print(f"Run geloggt: {anomalies4:,} Anomalien ({rate4*100:.1f}%)")
    print("Sichtbar unter: AI/ML → Experiments")


Run geloggt: 54,112 Anomalien (13.5%)
Sichtbar unter: AI/ML → Experiments
